# LOGO — Leave-One-Generator-Out (Kaggle)
For each held-out generator, trains MFFT-Base from scratch on all OTHER generators' fakes + real images, then evaluates on the held-out generator.

Measures cross-generator generalization (RQ4 / H3).

**Prerequisite:** `train_mfft_base.ipynb` must have completed (for manifest).

In [ ]:
# Cell 1: Clone repo & install deps
import subprocess, sys, shutil
from pathlib import Path

REPO_DIR = Path("/kaggle/working/mfft_repo")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", "https://github.com/MIHMahmudEli/ai-image-detection-research.git", str(REPO_DIR)], check=True)
sys.path.insert(0, str(REPO_DIR))
subprocess.run(["pip", "install", "-q", "huggingface_hub", "open_clip_torch", "scipy", "python-dotenv", "tqdm"], check=False)
print("Ready.")

In [ ]:
# Cell 1.5: Pre-flight mount check
!python /kaggle/working/mfft_repo/check_mounts.py

In [ ]:
# Cell 2: Setup
import os, json, time
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")

In [ ]:
# Cell 3: Load manifest & discover generators from shard metadata
from split_manifest_manager import SplitManifestManager

manifest_mgr = SplitManifestManager(hf_token=hf_token, run_id="logo")
manifest, manifest_sha256 = manifest_mgr.download()

# Group images by shard (each shard = one generator family)
# Map shards to generator names
SHARD_TO_GENERATOR = {
    "stable-diffusion": "Stable Diffusion",
    "midjourney": "Midjourney",
    "dall-e3": "DALL-E3",
    "genimage-ai": "BigGAN",
    "faceforensics": "FaceForensics",
    "dfdc-faces-of-the-train-sample": "DFDC",
    "celebdf-v2image-dataset": "Celeb-DF",
}

# All fake generators for LOGO
FAKE_GENERATORS = list(SHARD_TO_GENERATOR.values())
print(f"LOGO generators: {FAKE_GENERATORS}")
print(f"Manifest: {manifest['total_images']} images")

In [ ]:
# Cell 4: LOGO training function
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import GradScaler
from sklearn.metrics import f1_score, accuracy_score

LOGO_EPOCHS = 10

def train_logo_fold(held_out_gen, manifest, manifest_sha256, device):
    """Train on all generators EXCEPT held_out_gen, evaluate on held_out_gen's fakes + unseen reals."""
    print(f"\n{'='*60}")
    print(f"LOGO: held-out = {held_out_gen}")
    print(f"{'='*60}")

    # Split manifest entries into train/test
    # held_out_gen's fakes -> test
    # everything else -> train (reals + other generators' fakes)
    held_out_shard = None
    for shard, gen in SHARD_TO_GENERATOR.items():
        if gen == held_out_gen:
            held_out_shard = shard
            break

    train_entries = []
    test_fake_entries = []
    real_entries = []

    for img_id, info in manifest["images"].items():
        entry = {"image_id": img_id, **info}
        if info["label"] == "real":
            real_entries.append(entry)
        elif info["shard"] == held_out_shard:
            test_fake_entries.append(entry)
        else:
            train_entries.append(entry)

    # Split real entries: 70% train, 15% val, 15% test
    import random
    random.seed(42)
    random.shuffle(real_entries)
    n = len(real_entries)
    n_train = int(0.7 * n)
    n_val = int(0.15 * n)
    train_real = real_entries[:n_train]
    val_real = real_entries[n_train:n_train+n_val]
    test_real = real_entries[n_train+n_val:]

    train_data = train_entries + train_real
    val_data = val_real
    test_data = test_fake_entries + test_real

    print(f"  Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}")
    print(f"  Test fakes from: {held_out_gen} ({len(test_fake_entries)} images)")

    # Resolve paths and create DataLoaders
    from kaggle_dataset_loader import KaggleDatasetLoader
    # Build a mini-manifest for this fold
    fold_manifest = {
        "version": manifest["version"],
        "total_images": len(train_data) + len(val_data) + len(test_data),
        "class_distribution": manifest["class_distribution"],
        "split_sizes": {"train": len(train_data), "val": len(val_data), "test": len(test_data)},
        "images": {},
    }
    for e in train_data:
        fold_manifest["images"][e["image_id"]] = {**e, "split": "train"}
    for e in val_data:
        fold_manifest["images"][e["image_id"]] = {**e, "split": "val"}
    for e in test_data:
        fold_manifest["images"][e["image_id"]] = {**e, "split": "test"}

    m_loader = KaggleDatasetLoader(
        manifest=fold_manifest, manifest_sha256=manifest_sha256,
        input_root="/kaggle/input", image_size=224,
    )
    tr_loader, vl_loader, te_loader = m_loader.create_dataloaders(batch_size=64, num_workers=4)

    # Train MFFT-Base
    sys.path.insert(0, str(REPO_DIR / "model"))
    from src.model import build_mfft
    model = build_mfft("base").to(device)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)
    scheduler = CosineAnnealingLR(optimizer, T_max=LOGO_EPOCHS)
    scaler = GradScaler(enabled=torch.cuda.is_available())

    best_f1 = 0
    for epoch in range(1, LOGO_EPOCHS + 1):
        model.train()
        running_loss = 0.0
        n_batches = 0
        pbar = tqdm(tr_loader, desc=f"  Train {held_out_gen} {epoch:2d}/{LOGO_EPOCHS}", leave=False)
        for imgs, labels in pbar:
            imgs, labels = imgs.to(device), labels.to(device)
            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                loss = criterion(model(imgs), labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            running_loss += loss.item()
            n_batches += 1
            pbar.set_postfix(loss=f"{running_loss/n_batches:.4f}")
        scheduler.step()

        # Validate
        model.eval()
        preds, labels_all = [], []
        with torch.no_grad():
            for imgs, labels in tqdm(vl_loader, desc=f"  Val   {held_out_gen} {epoch:2d}/{LOGO_EPOCHS}", leave=False):
                imgs = imgs.to(device)
                preds.extend(model(imgs).argmax(1).cpu().numpy())
                labels_all.extend(labels.numpy())
        f1 = f1_score(labels_all, preds, average="macro", zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print(f"  Epoch {epoch}/{LOGO_EPOCHS} val_f1={f1:.4f}")

    # Test on held-out generator
    model.load_state_dict(best_state)
    model.eval()
    preds, labels_all = [], []
    with torch.no_grad():
        for imgs, labels in te_loader:
            imgs = imgs.to(device)
            preds.extend(model(imgs).argmax(1).cpu().numpy())
            labels_all.extend(labels.numpy())
    test_acc = accuracy_score(labels_all, preds)
    test_f1 = f1_score(labels_all, preds, average="macro", zero_division=0)

    print(f"  TEST ({held_out_gen}): acc={test_acc:.4f} f1={test_f1:.4f}")
    return {"held_out": held_out_gen, "test_acc": test_acc, "test_f1": test_f1}

In [ ]:
# Cell 5: Run LOGO across all generators
logo_results = []
for gen in FAKE_GENERATORS:
    try:
        r = train_logo_fold(gen, manifest, manifest_sha256, device)
        logo_results.append(r)
    except Exception as e:
        print(f"  FAILED {gen}: {e}")

# Summary
import pandas as pd
df = pd.DataFrame(logo_results)
print("\n=== LOGO Results ===")
print(df.to_string(index=False))
print(f"\nMean held-out accuracy: {df['test_acc'].mean():.4f}")
print(f"Mean held-out F1:       {df['test_f1'].mean():.4f}")

# Save & upload
out_dir = Path("/kaggle/working/logo_results")
out_dir.mkdir(exist_ok=True)
with open(out_dir / "logo_results.json", "w") as f:
    json.dump(logo_results, f, indent=2)
from huggingface_hub import HfApi
api = HfApi(token=hf_token)
api.upload_file(
    path_or_fileobj=str(out_dir / "logo_results.json"),
    path_in_repo="results/logo_results.json",
    repo_id="MohsinElis/mfft-checkpoints", repo_type="model",
)
print("Uploaded to HF.")

In [ ]:
# Cell 6: Plot LOGO results
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
df_sorted = df.sort_values("test_acc", ascending=True)
colors = ["#2ecc71" if a >= df["test_acc"].mean() else "#e74c3c" for a in df_sorted["test_acc"]]
ax.barh(df_sorted["held_out"], df_sorted["test_acc"], color=colors, edgecolor="white", height=0.6)
ax.set_xlabel("Test Accuracy")
ax.set_title("LOGO — Test Accuracy per Held-Out Generator")
ax.set_xlim(0, 1.0)
ax.axvline(df["test_acc"].mean(), color="black", linestyle="--", linewidth=1, label=f"Mean: {df['test_acc'].mean():.3f}")
ax.legend(loc="lower right")
for i, (_, row) in enumerate(df_sorted.iterrows()):
    ax.text(row["test_acc"] + 0.01, i, f"{row['test_acc']:.3f}", va="center", fontsize=9)
plt.tight_layout()
plt.savefig("/kaggle/working/logo_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to /kaggle/working/logo_results.png")